In [1]:
import finlab
from finlab import data

# 憑證已存於 ~/.finlab/credentials.json,已登入時不會再開瀏覽器
finlab.login()

已登入（使用快取憑證）。


In [2]:
# 只抓一份資料集做連線測試(有本機快取,重複執行不會重複消耗流量)
close = data.get("price:收盤價")
print("收盤價 shape:", close.shape)
close["2330"].tail()

收盤價 shape: (4790, 2786)


date
2026-10-12    2540.0
2026-10-13    2490.0
2026-10-14    2490.0
2026-10-15    2440.0
2026-10-16    2420.0
Name: 2330, dtype: float64

以下測試皆使用已載入的 `close`,不會再呼叫 API。

In [3]:
# 測試 1: 資料基本資訊(日期範圍、股票數、最新日期)
print("日期範圍:", close.index.min(), "~", close.index.max())
print("股票數:", close.shape[1])
print("最新一天有價格的股票數:", close.iloc[-1].notna().sum())

日期範圍: 2007-04-23 00:00:00 ~ 2026-10-16 00:00:00
股票數: 2786
最新一天有價格的股票數: 2357


In [4]:
# 測試 2: 比賽相關標的(0050、0056、2330、2317)最近 5 日收盤價
codes = [c for c in ["0050", "0056", "2330", "2317"] if c in close.columns]
print("存在的代號:", codes)
close[codes].tail()

存在的代號: ['0050', '0056', '2330', '2317']


symbol,0050,0056,2330,2317
date,,,,
2026-10-12,114.8,58.60,2540.0,244.5
2026-10-13,112.4,58.55,2490.0,240.0
2026-10-14,113.0,59.60,2490.0,243.0
2026-10-15,108.9,58.25,2440.0,239.5
2026-10-16,109.0,58.75,2420.0,244.0


In [5]:
# 測試 3: 漲跌幅(日報酬)與近 20 日累積報酬
ret = close[codes].pct_change()
print("最近 5 日日報酬:")
display(ret.tail().round(4))
print("近 20 日累積報酬:")
display((close[codes].iloc[-1] / close[codes].iloc[-21] - 1).round(4))

最近 5 日日報酬:


symbol,0050,0056,2330,2317
date,,,,
2026-10-12,-0.0013,0.0051,-0.0039,-0.0181
2026-10-13,-0.0209,-0.0009,-0.0197,-0.0184
2026-10-14,0.0053,0.0179,0.0000,0.0125
2026-10-15,-0.0363,-0.0227,-0.0201,-0.0144
2026-10-16,0.0009,0.0086,-0.0082,0.0188


近 20 日累積報酬:


symbol
0050    0.0259
0056    0.0682
2330    0.0147
2317   -0.0101
dtype: float64

In [6]:
# 測試 4: 簡單選股 — 今日收盤價創 60 日新高的股票
high60 = close.rolling(60).max()
new_high = close.iloc[-1] >= high60.iloc[-1]
print("創 60 日新高檔數:", int(new_high.sum()))
list(new_high[new_high].index[:20])

創 60 日新高檔數: 119


['0055',
 '00646',
 '00701',
 '00713',
 '00719B',
 '00731',
 '00849B',
 '00865B',
 '00878',
 '00897',
 '00907',
 '00915',
 '00916',
 '00926',
 '00932',
 '00938',
 '00939',
 '00949',
 '00961',
 '009814']

In [7]:
# 測試 5: 資料品質 — 缺值比例與是否有異常價格(<=0)
recent = close.tail(60)
print("近 60 日缺值比例:", round(float(recent.isna().mean().mean()), 4))
print("價格 <= 0 的筆數:", int((recent <= 0).sum().sum()))

近 60 日缺值比例: 0.1545
價格 <= 0 的筆數: 0
